In [68]:
import os
import sys
import pandas as pd
import re
import unicodedata
from pathlib import Path

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "AI").exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

PosixPath('/home/nynk/Documents/prj/clinic_management_system_ai')

In [38]:
BASE_DIR = Path.cwd().parents[1]
INPUT_FILE = (
    BASE_DIR
    / "training"
    / "dataset"
    / "disease"
    / "training"
    / "cleaned_disease_symptom_matrix.csv"
)
df = pd.read_csv(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (627, 2)
Columns: ['disease', 'symptoms']


,disease,symptoms
0,vocal cord polyp,"hoarse voice, sore throat, difficulty speaking..."
1,turner syndrome,"depression, emotional symptoms"
2,cryptorchidism,"lack of growth, symptoms of the scrotum and te..."
3,atrophic vaginitis,"suprapubic pain, vaginal itching, painful urin..."
4,cellulitis or abscess of mouth,"sore throat, throat swelling, difficulty in sw..."


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 627 entries, 0 to 626
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   disease   627 non-null    str  
 1   symptoms  627 non-null    str  
dtypes: str(2)
memory usage: 91.4 KB


In [15]:
df.isnull().sum()

disease     0
symptoms    0
dtype: int64

In [16]:
df["disease"].nunique()

627

In [18]:
df["symptoms"].nunique()

627

In [20]:
df["disease"].value_counts().head(20)

disease
vocal cord polyp                  1
turner syndrome                   1
cryptorchidism                    1
atrophic vaginitis                1
cellulitis or abscess of mouth    1
eye alignment disorder            1
pyloric stenosis                  1
salivary gland disorder           1
osteochondrosis                   1
metabolic disorder                1
vaginitis                         1
sick sinus syndrome               1
tinnitus of unknown cause         1
glaucoma                          1
transient ischemic attack         1
pyelonephritis                    1
liver cancer                      1
atelectasis                       1
choledocholithiasis               1
cirrhosis                         1
Name: count, dtype: int64

In [39]:
df["symptoms"] = df["symptoms"].fillna("")

df["symptoms"] = df["symptoms"].apply(
    lambda x: [
        symptom.strip()
        for symptom in x.split(",")
        if symptom.strip()
    ]
)

df.head()

,disease,symptoms
0,vocal cord polyp,"[hoarse voice, sore throat, difficulty speakin..."
1,turner syndrome,"[depression, emotional symptoms]"
2,cryptorchidism,"[lack of growth, symptoms of the scrotum and t..."
3,atrophic vaginitis,"[suprapubic pain, vaginal itching, painful uri..."
4,cellulitis or abscess of mouth,"[sore throat, throat swelling, difficulty in s..."


In [42]:
def normalize_text(text):
    text = str(text).strip().lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")

In [43]:
df["disease"] = df["disease"].apply(
    normalize_text
)

df["symptoms"] = df["symptoms"].apply(
    lambda symptoms: [
        normalize_text(symptom)
        for symptom in symptoms
    ]
)

df.head()

,disease,symptoms
0,vocal_cord_polyp,"[hoarse_voice, sore_throat, difficulty_speakin..."
1,turner_syndrome,"[depression, emotional_symptoms]"
2,cryptorchidism,"[lack_of_growth, symptoms_of_the_scrotum_and_t..."
3,atrophic_vaginitis,"[suprapubic_pain, vaginal_itching, painful_uri..."
4,cellulitis_or_abscess_of_mouth,"[sore_throat, throat_swelling, difficulty_in_s..."


In [44]:
df_exploded = df.explode(
    "symptoms"
).reset_index(drop=True)

df_exploded.head(20)

,disease,symptoms
0,vocal_cord_polyp,hoarse_voice
1,vocal_cord_polyp,sore_throat
2,vocal_cord_polyp,difficulty_speaking
3,vocal_cord_polyp,throat_swelling
4,vocal_cord_polyp,throat_feels_tight
5,turner_syndrome,depression
6,turner_syndrome,emotional_symptoms
7,cryptorchidism,lack_of_growth
8,cryptorchidism,symptoms_of_the_scrotum_and_testes
9,cryptorchidism,swelling_of_scrotum


In [45]:
disease_features = (
    df_exploded
    .assign(value=1)
    .pivot_table(
        index="disease",
        columns="symptoms",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

disease_features.columns.name = None

disease_features.head()

,disease,abdominal_distention,abnormal_appearing_skin,abnormal_breathing_sounds,abnormal_involuntary_movements,abnormal_movement_of_eyelid,absence_of_menstruation,abusing_alcohol,ache_all_over,acne_or_pimples,...,warts,weakness,weight_gain,wheezing,white_discharge_from_eye,wrinkles_on_skin,wrist_lump_or_mass,wrist_pain,wrist_swelling,wrist_weakness
0,abdominal_aortic_aneurysm,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,abdominal_hernia,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,abscess_of_nose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,abscess_of_the_lung,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,abscess_of_the_pharynx,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [51]:
from pathlib import Path

OUTPUT_DIR = Path("../dataset/disease")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTPUT_DIR / "disease_features.csv"

disease_features.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", OUTPUT.resolve())

Saved: /home/nynk/Documents/prj/clinic_management_system_ai/training/dataset/disease/disease_features.csv


In [52]:
print("Diseases:", disease_features["disease"].nunique())
print("Symptoms:", len(disease_features.columns) - 1)

print(
    disease_features.columns.tolist()
)

Diseases: 627
Symptoms: 322
['disease', 'abdominal_distention', 'abnormal_appearing_skin', 'abnormal_breathing_sounds', 'abnormal_involuntary_movements', 'abnormal_movement_of_eyelid', 'absence_of_menstruation', 'abusing_alcohol', 'ache_all_over', 'acne_or_pimples', 'allergic_reaction', 'ankle_pain', 'ankle_swelling', 'anxiety_and_nervousness', 'apnea', 'arm_lump_or_mass', 'arm_pain', 'arm_stiffness_or_tightness', 'arm_swelling', 'arm_weakness', 'back_cramps_or_spasms', 'back_mass_or_lump', 'back_pain', 'back_stiffness_or_tightness', 'back_weakness', 'bedwetting', 'bladder_mass', 'bleeding_from_ear', 'bleeding_from_eye', 'bleeding_gums', 'bleeding_or_discharge_from_nipple', 'blindness', 'blood_clots_during_menstrual_periods', 'blood_in_stool', 'blood_in_urine', 'bones_are_painful', 'bowlegged_or_knock_kneed', 'bumps_on_penis', 'burning_abdominal_pain', 'burning_chest_pain', 'changes_in_stool_appearance', 'chest_tightness', 'chills', 'cloudy_eye', 'congestion_in_chest', 'constipation', 

In [53]:
print(disease_features.shape)

print(
    disease_features["disease"].head(20).tolist()
)

print(
    disease_features.iloc[:, 1:].sum().describe()
)

(627, 323)
['abdominal_aortic_aneurysm', 'abdominal_hernia', 'abscess_of_nose', 'abscess_of_the_lung', 'abscess_of_the_pharynx', 'acanthosis_nigricans', 'acariasis', 'achalasia', 'acne', 'actinic_keratosis', 'acute_bronchiolitis', 'acute_bronchitis', 'acute_bronchospasm', 'acute_fatty_liver_of_pregnancy_aflp', 'acute_glaucoma', 'acute_kidney_injury', 'acute_otitis_media', 'acute_pancreatitis', 'acute_respiratory_distress_syndrome_ards', 'acute_sinusitis']
count    322.000000
mean      13.897516
std       17.633480
min        1.000000
25%        3.000000
50%        8.000000
75%       19.000000
max      124.000000
dtype: float64


In [56]:
disease_features["symptom_count"] = (
    disease_features.iloc[:, 1:].sum(axis=1)
)

print(
    disease_features["symptom_count"].describe()
)

count    627.000000
mean       7.137161
std        3.092866
min        2.000000
25%        5.000000
50%        7.000000
75%       10.000000
max       15.000000
Name: symptom_count, dtype: float64


In [57]:
disease_features[
    ["disease", "symptom_count"]
].sort_values(
    "symptom_count",
    ascending=False
).head(20)

,disease,symptom_count
211,fungal_infection_of_the_hair,15
175,eczema,15
519,skin_pigmentation_disorder,14
134,contact_dermatitis,14
9,actinic_keratosis,14
471,psoriasis,14
505,seborrheic_dermatitis,14
520,skin_polyp,14
518,skin_disorder,13
299,ingrown_toe_nail,13


In [63]:
import pandas as pd
from pathlib import Path

# Đọc disease_features
INPUT = Path("../dataset/disease/disease_features.csv")

disease_features = pd.read_csv(INPUT)

print("Input shape:", disease_features.shape)

Input shape: (627, 323)


In [64]:
disease_kb = (
    disease_features
    .set_index("disease")
    .stack()
    .reset_index()
)

disease_kb.columns = [
    "disease",
    "symptom",
    "present"
]

# Chỉ giữ symptom tồn tại trong disease
disease_kb = disease_kb[
    disease_kb["present"] == 1
].drop(
    columns=["present"]
)

disease_kb = disease_kb.sort_values(
    ["disease", "symptom"]
).reset_index(drop=True)

print(disease_kb.head(20))
print("Relations:", len(disease_kb))

                      disease                             symptom
0   abdominal_aortic_aneurysm                        arm_swelling
1   abdominal_aortic_aneurysm                           back_pain
2   abdominal_aortic_aneurysm              burning_abdominal_pain
3   abdominal_aortic_aneurysm                        palpitations
4   abdominal_aortic_aneurysm                sharp_abdominal_pain
5   abdominal_aortic_aneurysm                 shortness_of_breath
6            abdominal_hernia                       ache_all_over
7            abdominal_hernia                          groin_mass
8            abdominal_hernia              irregular_belly_button
9            abdominal_hernia                lower_abdominal_pain
10           abdominal_hernia                       regurgitation
11           abdominal_hernia                     regurgitation_1
12           abdominal_hernia                sharp_abdominal_pain
13           abdominal_hernia                     swollen_abdomen
14        

In [65]:
OUTPUT = Path(
    "../dataset/disease/disease_knowledge_base.csv"
)

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

disease_kb.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", OUTPUT.resolve())

Saved: /home/nynk/Documents/prj/clinic_management_system_ai/training/dataset/disease/disease_knowledge_base.csv


In [69]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "AI").exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from AI.disease.knowledge_base import DiseaseKnowledgeBase

kb = DiseaseKnowledgeBase(
    project_root / "training" / "dataset" / "disease" / "disease_knowledge_base.csv"
)

kb